# Présentation de la Base de Données Olist

Olist est une marketplace brésilienne qui met en relation des vendeurs indépendants et des acheteurs. Le dataset couvre **~100 000 commandes** passées entre 2016 et 2018 sur l'ensemble du territoire brésilien.

La base est organisée en **9 tables relationnelles** qui décrivent chaque étape du cycle de vie d'une commande.

---

### Schéma relationnel

```
olist_customers ──────────────────────────────────────────────┐
    customer_id                                               │
        │                                                     │
        ▼                                                     │
  olist_orders ──── olist_order_items ──── olist_products     │
    order_id              order_id              product_id    │
        │                    │                               │
        │                    └──────── olist_sellers         │
        │                                 seller_id          │
        ├──── olist_order_payments                           │
        │         order_id                                   │
        └──── olist_order_reviews                            │
                  order_id                                   │
                                                             │
olist_geolocation ◄── zip_code_prefix ───────────────────────┘
product_category_name_translation ◄── product_category_name
```

**Unité d'analyse :** `customer_unique_id` — identifiant pérenne du client, indépendant de ses adresses de livraison successives.

In [5]:
import sys, os, subprocess, time
from pathlib import Path
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
from sqlalchemy import create_engine

project_root = Path().resolve().parent
sys.path.append(str(project_root))
load_dotenv(project_root / ".env")

PG_CONTAINER = "olist_postgres"

if subprocess.run(
    ["docker", "ps", "--filter", f"name={PG_CONTAINER}", "--format", "{{.Names}}"],
    capture_output=True, text=True
).stdout.strip():
    print(f"Conteneur '{PG_CONTAINER}' actif.")
else:
    subprocess.run(["docker", "start", PG_CONTAINER], check=True)
    time.sleep(3)
    print("Conteneur redemarre.")

engine = create_engine(
    f"postgresql://"
    f"{os.getenv('POSTGRES_USER', 'olist_user')}:"
    f"{os.getenv('POSTGRES_PASSWORD', 'olist_password')}@"
    f"{os.getenv('POSTGRES_HOST', 'localhost')}:"
    f"{os.getenv('POSTGRES_PORT', '5444')}/"
    f"{os.getenv('POSTGRES_DB', 'olist_db')}"
)

tables = [
    'olist_orders', 'olist_customers', 'olist_order_items',
    'olist_products', 'olist_sellers', 'olist_order_payments',
    'olist_order_reviews', 'olist_geolocation',
    'product_category_name_translation'
]
db_data = {t: pd.read_sql_table(t, engine) for t in tables}
print('Tables chargees :')
for t, df in db_data.items():
    print(f'  {t:<45} {len(df):>8,} lignes x {df.shape[1]} colonnes')


Conteneur 'olist_postgres' actif.
Tables chargees :
  olist_orders                                    99,441 lignes x 8 colonnes
  olist_customers                                 99,441 lignes x 5 colonnes
  olist_order_items                              112,650 lignes x 7 colonnes
  olist_products                                  32,951 lignes x 9 colonnes
  olist_sellers                                    3,095 lignes x 4 colonnes
  olist_order_payments                           103,886 lignes x 5 colonnes
  olist_order_reviews                             99,224 lignes x 7 colonnes
  olist_geolocation                             1,000,163 lignes x 5 colonnes
  product_category_name_translation                   71 lignes x 2 colonnes


---
## Table 1 — `olist_orders`
**Rôle :** Table centrale du schéma. Chaque ligne représente **une commande**, avec son statut et ses horodatages clés.

| Colonne | Description |
|---------|-------------|
| `order_id` | Identifiant unique de la commande — clé primaire |
| `customer_id` | Identifiant de livraison du client (peut changer à chaque commande) |
| `order_status` | Statut : `delivered`, `shipped`, `canceled`, `invoiced`, `processing`… |
| `order_purchase_timestamp` | Date et heure de passage de la commande |
| `order_approved_at` | Date de validation du paiement |
| `order_delivered_carrier_date` | Date de remise au transporteur |
| `order_delivered_customer_date` | Date de livraison effective chez le client |
| `order_estimated_delivery_date` | Date de livraison estimée communiquée au client |

> **Usage projet :** `order_purchase_timestamp` → Recency. `order_delivered_customer_date - order_estimated_delivery_date` → `avg_delivery_delay`.

In [6]:
df = db_data['olist_orders']
print(f"olist_orders : {len(df):,} lignes x {df.shape[1]} colonnes")
print(f"Statuts : {df['order_status'].value_counts().to_dict()}")
display(df.head())

olist_orders : 99,441 lignes x 8 colonnes
Statuts : {'delivered': 96478, 'shipped': 1107, 'canceled': 625, 'unavailable': 609, 'invoiced': 314, 'processing': 301, 'created': 5, 'approved': 2}


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


---
## Table 2 — `olist_customers`
**Rôle :** Référentiel client avec localisation géographique.

| Colonne | Description |
|---------|-------------|
| `customer_id` | Identifiant de livraison — clé étrangère vers `olist_orders` |
| `customer_unique_id` | **Identifiant pérenne du client** — unité d'analyse du projet |
| `customer_zip_code_prefix` | Code postal (5 chiffres) |
| `customer_city` | Ville du client |
| `customer_state` | État brésilien (2 lettres) : SP, RJ, MG… |

> **Usage projet :** Agrégation de toutes les commandes au niveau `customer_unique_id`. `customer_state` → `region_freight_score`.

In [7]:
df = db_data['olist_customers']
print(f"olist_customers : {len(df):,} lignes x {df.shape[1]} colonnes")
print(f"Clients uniques (customer_unique_id) : {df['customer_unique_id'].nunique():,}")
print(f"Etats couverts : {df['customer_state'].nunique()}")
display(df.head())

olist_customers : 99,441 lignes x 5 colonnes
Clients uniques (customer_unique_id) : 96,096
Etats couverts : 27


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


---
## Table 3 — `olist_order_items`
**Rôle :** Détail des articles de chaque commande. Une commande peut contenir plusieurs articles → plusieurs lignes par `order_id`.

| Colonne | Description |
|---------|-------------|
| `order_id` | Identifiant de la commande — clé étrangère |
| `order_item_id` | Numéro de l'article dans la commande (1, 2, 3…) |
| `product_id` | Identifiant du produit — clé étrangère vers `olist_products` |
| `seller_id` | Identifiant du vendeur — clé étrangère vers `olist_sellers` |
| `shipping_limit_date` | Date limite d'expédition imposée au vendeur |
| `price` | Prix unitaire de l'article (BRL) |
| `freight_value` | Frais de port de l'article (BRL) |

> **Usage projet :** `freight_value / price` → `avg_freight_ratio`. `sum(price + freight_value)` par client → `Monetary`.

In [8]:
df = db_data['olist_order_items']
print(f"olist_order_items : {len(df):,} lignes x {df.shape[1]} colonnes")
print(f"Commandes distinctes : {df['order_id'].nunique():,}")
print(f"Prix moyen : {df['price'].mean():.2f} BRL | Frais de port moyens : {df['freight_value'].mean():.2f} BRL")
display(df.head())

olist_order_items : 112,650 lignes x 7 colonnes
Commandes distinctes : 98,666
Prix moyen : 120.65 BRL | Frais de port moyens : 19.99 BRL


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


---
## Table 4 — `olist_products`
**Rôle :** Catalogue produits avec catégorie et caractéristiques physiques.

| Colonne | Description |
|---------|-------------|
| `product_id` | Identifiant unique du produit — clé primaire |
| `product_category_name` | Catégorie en portugais (ex: `cama_mesa_banho`) |
| `product_name_lenght` | Longueur du nom du produit (nb de caractères) |
| `product_description_lenght` | Longueur de la description (nb de caractères) |
| `product_photos_qty` | Nombre de photos du produit |
| `product_weight_g` | Poids en grammes |
| `product_length_cm` | Longueur en cm |
| `product_height_cm` | Hauteur en cm |
| `product_width_cm` | Largeur en cm |

> **Usage projet :** `product_category_name` (traduit) → `category_tier_encoded` et recommandations produits par cluster (Section 9b).

In [9]:
df = db_data['olist_products']
print(f"olist_products : {len(df):,} lignes x {df.shape[1]} colonnes")
print(f"Categories distinctes : {df['product_category_name'].nunique()}")
print(f"Top 5 categories :")
print(df['product_category_name'].value_counts().head().to_string())
display(df.head())

olist_products : 32,951 lignes x 9 colonnes
Categories distinctes : 73
Top 5 categories :
product_category_name
cama_mesa_banho          3029
esporte_lazer            2867
moveis_decoracao         2657
beleza_saude             2444
utilidades_domesticas    2335


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


---
## Table 5 — `olist_sellers`
**Rôle :** Référentiel des vendeurs partenaires Olist avec leur localisation.

| Colonne | Description |
|---------|-------------|
| `seller_id` | Identifiant unique du vendeur — clé primaire |
| `seller_zip_code_prefix` | Code postal du vendeur |
| `seller_city` | Ville du vendeur |
| `seller_state` | État brésilien du vendeur |

> **Usage projet :** Analyse de la couverture géographique des vendeurs et impact sur les délais de livraison selon la distance vendeur-client.

In [10]:
df = db_data['olist_sellers']
print(f"olist_sellers : {len(df):,} lignes x {df.shape[1]} colonnes")
print(f"Etats couverts par les vendeurs : {sorted(df['seller_state'].unique())}")
display(df.head())

olist_sellers : 3,095 lignes x 4 colonnes
Etats couverts par les vendeurs : ['AC', 'AM', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RS', 'SC', 'SE', 'SP']


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


---
## Table 6 — `olist_order_payments`
**Rôle :** Détail des paiements par commande. Une commande peut avoir plusieurs lignes si le client combine plusieurs modes de paiement.

| Colonne | Description |
|---------|-------------|
| `order_id` | Identifiant de la commande — clé étrangère |
| `payment_sequential` | Numéro du paiement si combinaison de modes (1, 2…) |
| `payment_type` | Mode : `credit_card`, `boleto`, `voucher`, `debit_card` |
| `payment_installments` | Nombre de mensualités (1 = paiement comptant) |
| `payment_value` | Montant de ce paiement (BRL) |

> **Usage projet :** `mode(payment_type) == credit_card` → `payment_type_cc_flag`. `mean(payment_installments)` → `avg_installments`. `sum(payment_value)` → `Monetary`.

> **Boleto :** Équivalent brésilien du virement bancaire, très utilisé sans carte bancaire — proxy de revenu modeste.

In [11]:
df = db_data['olist_order_payments']
print(f"olist_order_payments : {len(df):,} lignes x {df.shape[1]} colonnes")
print(f"Repartition des modes de paiement :")
print(df['payment_type'].value_counts().to_string())
print(f"\nNb de versements moyen : {df['payment_installments'].mean():.1f}")
display(df.head())

olist_order_payments : 103,886 lignes x 5 colonnes
Repartition des modes de paiement :
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3

Nb de versements moyen : 2.9


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


---
## Table 7 — `olist_order_reviews`
**Rôle :** Avis clients laissés après livraison. Note de 1 (très insatisfait) à 5 (très satisfait).

| Colonne | Description |
|---------|-------------|
| `review_id` | Identifiant unique de l'avis |
| `order_id` | Identifiant de la commande évaluée — clé étrangère |
| `review_score` | Note de 1 à 5 étoiles |
| `review_comment_title` | Titre du commentaire (souvent vide) |
| `review_comment_message` | Texte du commentaire en portugais (souvent vide) |
| `review_creation_date` | Date d'envoi de la demande d'avis |
| `review_answer_timestamp` | Date de réponse du client |

> **Usage projet :** `mean(review_score)` par client → `avg_review_score`, indicateur de satisfaction et de fidélisabilité du segment.

In [12]:
df = db_data['olist_order_reviews']
print(f"olist_order_reviews : {len(df):,} lignes x {df.shape[1]} colonnes")
print(f"Repartition des notes :")
print(df['review_score'].value_counts().sort_index().to_string())
print(f"Note moyenne globale : {df['review_score'].mean():.2f} / 5")
display(df.head())

olist_order_reviews : 99,224 lignes x 7 colonnes
Repartition des notes :
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Note moyenne globale : 4.09 / 5


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,None,None,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,None,None,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,None,None,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,None,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,None,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


---
## Table 8 — `olist_geolocation`
**Rôle :** Coordonnées GPS associées à chaque code postal brésilien.

| Colonne | Description |
|---------|-------------|
| `geolocation_zip_code_prefix` | Code postal (5 chiffres) — clé de jointure |
| `geolocation_lat` | Latitude (degrés décimaux) |
| `geolocation_lng` | Longitude (degrés décimaux) |
| `geolocation_city` | Ville correspondante |
| `geolocation_state` | État brésilien |

> **Usage projet :** Visualisations cartographiques (choroplèthe des états) et construction de `region_freight_score`. Un même code postal peut avoir plusieurs entrées GPS — on prend la médiane.

In [13]:
df = db_data['olist_geolocation']
print(f"olist_geolocation : {len(df):,} lignes x {df.shape[1]} colonnes")
print(f"Codes postaux uniques : {df['geolocation_zip_code_prefix'].nunique():,}")
print(f"Latitude  : {df['geolocation_lat'].min():.2f} -> {df['geolocation_lat'].max():.2f}")
print(f"Longitude : {df['geolocation_lng'].min():.2f} -> {df['geolocation_lng'].max():.2f}")
display(df.head())

olist_geolocation : 1,000,163 lignes x 5 colonnes
Codes postaux uniques : 19,015
Latitude  : -36.61 -> 45.07
Longitude : -101.47 -> 121.11


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


---
## Table 9 — `product_category_name_translation`
**Rôle :** Table de correspondance entre les noms de catégories en portugais et leur traduction en anglais.

| Colonne | Description |
|---------|-------------|
| `product_category_name` | Nom en portugais — clé de jointure vers `olist_products` |
| `product_category_name_english` | Traduction en anglais |

> **Exemple :** `cama_mesa_banho` → `bed_bath_table` | `esporte_lazer` → `sports_leisure`

> **Usage projet :** Jointure systématique pour afficher des labels lisibles dans l'EDA, le dashboard et les recommandations produits.

In [14]:
df = db_data['product_category_name_translation']
print(f"product_category_name_translation : {len(df):,} lignes x {df.shape[1]} colonnes")
display(df.head(10))

product_category_name_translation : 71 lignes x 2 colonnes


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
5,esporte_lazer,sports_leisure
6,perfumaria,perfumery
7,utilidades_domesticas,housewares
8,telefonia,telephony
9,relogios_presentes,watches_gifts


---
## Table Maître — `df_master`
**Rôle :** DataFrame consolidé par jointure de toutes les tables. Table de travail principale pour le feature engineering.

**Grain :** 1 ligne = 1 article commandé, avec toutes les informations client, commande, produit et paiement associées.

```
olist_orders
    JOIN olist_customers      ON customer_id
    JOIN olist_order_items    ON order_id
    JOIN olist_products       ON product_id
    JOIN olist_sellers        ON seller_id
    JOIN olist_order_payments ON order_id
    LEFT JOIN olist_order_reviews ON order_id
    JOIN product_category_name_translation ON product_category_name
```

> Après jointure, on agrège au niveau `customer_unique_id` pour construire une **vue client unique** et calculer les features RFM.

In [15]:
df_master = get_merged_dataframe(engine)
print(f"df_master : {len(df_master):,} lignes x {df_master.shape[1]} colonnes")
print(f"\nColonnes et taux de nulls :")
for col in df_master.columns:
    pct = df_master[col].isna().mean() * 100
    print(f"  {col:<45} {str(df_master[col].dtype):<12} {pct:>5.1f}% nulls")
print()
display(df_master.head())

2026-05-08 16:43:42,485 - INFO - Fetching and merging data from PostgreSQL...
2026-05-08 16:43:43,480 - INFO - Successfully fetched 115723 rows.


df_master : 115,723 lignes x 35 colonnes

Colonnes et taux de nulls :
  order_id                                      object         0.0% nulls
  customer_id                                   object         0.0% nulls
  order_status                                  object         0.0% nulls
  order_purchase_timestamp                      object         0.0% nulls
  order_approved_at                             object         0.0% nulls
  order_delivered_carrier_date                  object         0.0% nulls
  order_delivered_customer_date                 object         0.0% nulls
  order_estimated_delivery_date                 object         0.0% nulls
  customer_unique_id                            object         0.0% nulls
  customer_zip_code_prefix                      int64          0.0% nulls
  customer_city                                 object         0.0% nulls
  customer_state                                object         0.0% nulls
  order_item_id                           

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,seller_city,seller_state,payment_sequential,payment_type,payment_installments,payment_value,review_id,review_score,review_creation_date,review_answer_timestamp
0,6ebaec694d7025e2ad4a05dba887c032,4f28355e5c17a4a42d3ce2439a1d4501,delivered,2017-05-18 13:55:47,2017-05-18 14:05:17,2017-05-19 12:01:38,2017-05-29 12:47:20,2017-06-09 00:00:00,4acce2834231e13b1514915adda5ec2b,21910,...,cariacica,ES,1.0,credit_card,7.0,153.72,590aec3016a098ad236fc59459173f33,1.0,2017-05-30 00:00:00,2017-05-31 13:34:59
1,138849fd84dff2fb4ca70a0a34c4aa1c,9b18f3fc296990b97854e351334a32f6,delivered,2018-02-01 14:02:19,2018-02-03 02:53:07,2018-02-06 19:13:26,2018-02-14 13:41:59,2018-02-23 00:00:00,b2cac0b16835dabf811b204127f58afa,6330,...,capivari,SP,1.0,boleto,1.0,52.84,16fe408521a9c6de156276d366356476,5.0,2018-02-15 00:00:00,2018-02-15 17:07:02
2,68873cf91053cd11e6b49a766db5af1a,4632eb5a8f175f6fe020520ae0c678f3,delivered,2017-11-30 22:02:15,2017-12-02 02:51:18,2017-12-04 22:07:01,2017-12-05 20:28:40,2017-12-18 00:00:00,6da92ae920ab16fc4eceb8fcd7bd43ce,8280,...,sao paulo,SP,1.0,boleto,1.0,91.66,49dc1a77d03e598dbe358fbdd36d602c,4.0,2017-12-15 00:00:00,2017-12-17 19:12:44
3,f346ad4ee8f630e5e4ddaf862a34e6dd,dd5095632e3953fc0947b8ab5176b0be,delivered,2018-08-05 13:09:48,2018-08-05 13:24:34,2018-08-06 13:41:00,2018-08-10 18:35:40,2018-08-15 00:00:00,da45a9a1df408c39f013b9b0b505042c,70680,...,goiania,GO,1.0,credit_card,1.0,53.66,a2564068ec090aa597394b0d29c95e2a,5.0,2018-08-11 00:00:00,2018-08-14 00:34:02
4,ccbabeb0b02433bd0fcbac46e70339f2,c77ee2d8ba1614a4d489a44166894938,delivered,2018-02-19 20:31:09,2018-02-21 06:15:25,2018-02-22 21:04:23,2018-03-09 22:22:25,2018-03-13 00:00:00,9c9cef121cb812cb301babddc2d8331e,38067,...,atibaia,SP,1.0,boleto,1.0,43.00,8972e42b54449efa29204423f1a9ead7,4.0,2018-03-10 00:00:00,2018-03-11 21:08:38


---
## Récapitulatif

| Table | Lignes approx. | Colonnes | Rôle |
|-------|----------------|----------|------|
| `olist_orders` | ~99 000 | 8 | Table centrale — statuts et dates |
| `olist_customers` | ~99 000 | 5 | Référentiel client + localisation |
| `olist_order_items` | ~112 000 | 7 | Articles, prix, frais de port |
| `olist_products` | ~33 000 | 9 | Catalogue produits |
| `olist_sellers` | ~3 000 | 4 | Référentiel vendeurs |
| `olist_order_payments` | ~103 000 | 5 | Paiements et modes de règlement |
| `olist_order_reviews` | ~99 000 | 7 | Avis clients |
| `olist_geolocation` | ~1 000 000 | 5 | GPS par code postal |
| `product_category_name_translation` | 71 | 2 | Traduction catégories PT → EN |
| **`df_master`** | **~112 000** | **~30** | **Table de travail consolidée** |